# Candidate generation для поиска услуг Авито

Решение объединяет BM25, поиск по обучающим запросам и CatBoostRanker.
Настройки находятся в `configs/improved.toml`, реализация этапов в `src/avito_improved/`.

При `rebuild = False` показываются сохранённые результаты. Для полного расчёта положите три Parquet в `data/`, установите `rebuild = True` и выполните все ячейки.

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time

os.environ.update({"POLARS_MAX_THREADS": "2", "OPENBLAS_NUM_THREADS": "2", "OMP_NUM_THREADS": "2"})

from avito_ranker.config import load_config
from avito_improved.run import run_stage

project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
config_path = project_dir / "configs/improved.toml"
config = load_config(config_path)
work = config["work_dir"] / "quality"
saved = config["results_dir"]

rebuild = False
{"rebuild": rebuild, "config": str(config_path.name)}

{'rebuild': False, 'config': 'improved.toml'}

## 1. Проверки кода

In [2]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=project_dir, check=True)

CompletedProcess(args=['D:\\Codex_artifacts\\Avito_test\\layout_verification\\environment\\Scripts\\python.exe', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], returncode=0)

## 2. Данные и разбиение

Группы нормализованных запросов не пересекаются между train и dev. История строится только на train.
Из ранее не использовавшихся holdout-групп заранее выбираются 1 000 запросов для итоговой оценки. Их метки не участвуют в поиске или выборе модели.

In [3]:
if rebuild:
    run_stage("prepare", config_path)

## 3. Индексы локального корпуса

Корпус содержит все доступные объявления из train и benchmark. Он выбирается независимо от положительных меток.

In [4]:
if rebuild:
    run_stage("validation_indices", config_path)

## 4. Обучающие пары

Для каждого запроса ищутся кандидаты, затем добавляются метки из train. В истории исключается вся группа самого обучающего запроса.
Исходная модель `results/ranker.cbm` помогает выбрать сложные отрицательные примеры; этот фиксированный файл входит в репозиторий.

In [5]:
if rebuild:
    run_stage("train_features", config_path)
    run_stage("audit", config_path)

## 5. Признаки dev

Положительные объявления, не попавшие в пул кандидатов, остаются в знаменателе Recall@50.

In [6]:
if rebuild:
    run_stage("dev_features", config_path)

## 6. Обучение

Сравниваются PairLogit и QuerySoftMax. Для каждой модели проверяются от 20 до 400 деревьев с шагом 20. Выбор делается по среднему Recall@50 на dev.

In [7]:
if rebuild:
    run_stage("train", config_path)

## 7. Дополнительные кандидаты и объединение выдач

На dev сравниваются поиск без расширения и добавление BM25-кандидатов в радиусе 50 км. Также проверяется дополнение топа новой модели объявлениями исходной выдачи. В ответе остаются 50 разных ID.

In [8]:
if rebuild:
    run_stage("dev_expanded_features", config_path)
    run_stage("select", config_path)
selection_path = work / "selected.json" if rebuild else saved / "selected.json"
json.loads(selection_path.read_text("utf-8"))

{'method': 'pairlogit',
 'trees': 80,
 'recall_50': 0.7832083333333333,
 'ranker_recall_50': 0.7772916666666666,
 'nearby_candidates': True,
 'new_head': 40,
 'blend_options': [{'nearby_candidates': False,
   'new_head': 0,
   'recall_50': 0.7410333333333333},
  {'nearby_candidates': False,
   'new_head': 30,
   'recall_50': 0.7809083333333333},
  {'nearby_candidates': False,
   'new_head': 40,
   'recall_50': 0.7809583333333333},
  {'nearby_candidates': False,
   'new_head': 45,
   'recall_50': 0.7792083333333333},
  {'nearby_candidates': False,
   'new_head': 50,
   'recall_50': 0.7772916666666666},
  {'nearby_candidates': True, 'new_head': 0, 'recall_50': 0.7410333333333333},
  {'nearby_candidates': True, 'new_head': 30, 'recall_50': 0.7821583333333333},
  {'nearby_candidates': True, 'new_head': 40, 'recall_50': 0.7832083333333333},
  {'nearby_candidates': True, 'new_head': 45, 'recall_50': 0.7814583333333333},
  {'nearby_candidates': True,
   'new_head': 50,
   'recall_50': 0.77854

## 8. Фиксация

До открытия нового holdout сохраняются контрольные суммы модели, кода и обучающей истории.

In [9]:
if rebuild:
    run_stage("freeze", config_path)

## 9. Итоговая проверка

Выбранное решение и исходная модель сравниваются на одних 1 000 запросах. После этой оценки параметры не подбираются.

In [10]:
if rebuild:
    run_stage("fresh_features", config_path)
    run_stage("evaluate", config_path)
metrics_path = work / "fresh_metrics.json" if rebuild else saved / "fresh_metrics.json"
json.loads(metrics_path.read_text("utf-8"))

{'queries': 1000,
 'original_recall_50': 0.7358333333333333,
 'bm25_rrf_recall_50': 0.2788333333333334,
 'improved_recall_50': 0.7848333333333334,
 'delta': 0.049,
 'delta_ci95': [0.030833333333333334, 0.06700416666666664],
 'pool_recall': 0.935,
 'model_sha256': '3107ea0b82c189e86e1ee524e6fd65d9b8cb1430aa2a269054c7d0060628ccee',
 'selected_before_fresh_evaluation': True}

## 10. Поиск в benchmark

In [11]:
if rebuild:
    run_stage("benchmark_indices", config_path)
    run_stage("benchmark_features", config_path)

## 11. Формирование CSV

In [12]:
if rebuild:
    run_stage("predict", config_path)

## 12. Сохранение результатов

`answer.csv` содержит только `query_id` и `answer`. В каждой строке 50 уникальных идентификаторов из benchmark. Код проверки доступен в `src/avito_retrieval/submission.py`.

In [13]:
import shutil

if rebuild:
    run_stage("export", config_path)
    shutil.copyfile(saved / "answer.csv", project_dir / "answer.csv")
json.loads((saved / "submission_check.json").read_text("utf-8"))

{'valid': True,
 'rows': 2452,
 'min_items': 50,
 'max_items': 50,
 'sha256': '88cf3ad8249ff7a3baf14759655df3d74eab0863193db6c55930d4a525c5fb64',
 'model_sha256': '3107ea0b82c189e86e1ee524e6fd65d9b8cb1430aa2a269054c7d0060628ccee'}

Обучение и вычисления выполняются локально, без внешних API. Использованные библиотеки и команды установки перечислены в [README](../README.md).